In [5]:
%pip install numpy pandas duckdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 54.9 MB/s  0:00:006m0:00:01

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import duckdb

conn = duckdb.connect("main.db")

In [3]:
conn.execute("""
CREATE TABLE IF NOT EXISTS air_purifier_search_trend AS
SELECT *
FROM read_csv(
    '/workspaces/Product-Market-Fit-Analysis---Case-Study/Dataset/air_purifier_search_trend_data.csv',
    ignore_errors=true
             )
""")
conn.execute("SELECT * FROM air_purifier_search_trend LIMIT 5").fetchdf()

,Time,air purifier
0,2004-01-01,0
1,2004-02-01,0
2,2004-03-01,0
3,2004-04-01,0
4,2004-05-01,0


In [18]:
# Which pollutants appear most frequently as "prominent pollutants"?

conn.execute(""" 
WITH pollutants_unnested AS (
SELECT
    date,
    state,
    area,
    air_quality_status,
    aqi_value,
    UNNEST(STRING_SPLIT(prominent_pollutants, ',')) AS pollutants        
FROM air_quality
)

SELECT
    area,
    pollutants,
    AVG(aqi_value) AS avg_aqi,
    COUNT(DISTINCT date) AS pollutant_occurence
FROM pollutants_unnested
GROUP BY area, pollutants
ORDER BY pollutant_occurence DESC

""").fetch_df()

,area,pollutants,avg_aqi,pollutant_occurence
0,Lucknow,PM2.5,189.285668,3098
1,Delhi,PM2.5,234.621640,2902
2,Faridabad,PM2.5,217.503471,2737
3,Hyderabad,PM10,84.758633,2722
4,Bengaluru,PM10,75.121269,2647
...,...,...,...,...
1561,Jind,SO2,38.000000,1
1562,Belapur,NH3,26.000000,1
1563,Vijayapura,NO2,60.000000,1
1564,Bhiwadi,SO2,28.000000,1


In [29]:
# Which cities consistently show Poor/Very Poor/Severe AQI?

conn.execute(""" 
SELECT
    area,
    air_quality_status,
    COUNT(DISTINCT date) AS num_of_days,
    AVG(aqi_value) AS avg_aqi
FROM air_quality
WHERE air_quality_status IN ('Poor','Very Poor', 'Severe')
GROUP BY area, air_quality_status
ORDER BY area, num_of_days DESC
""").fetch_df()

,area,air_quality_status,num_of_days,avg_aqi
0,Agartala,Poor,319,242.708464
1,Agartala,Very Poor,46,320.717391
2,Agra,Poor,430,248.590698
3,Agra,Very Poor,303,348.910891
4,Agra,Severe,86,427.372093
...,...,...,...,...
588,Vrindavan,Very Poor,11,335.818182
589,Vrindavan,Severe,7,452.142857
590,Yadgir,Poor,9,217.000000
591,Yamunanagar,Poor,397,244.652393
